# Synthetic Dataset: 3 Algorithms + Benchmark (через `src/flowopt`)

Ноутбук сравнивает 3 алгоритма на синтетическом датасете:
- `GAP + VRP`
- `MILP`
- `Genetic`

Выход: единая benchmark-таблица по качеству и времени.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from flowopt.pipeline import DATA_ROOT, benchmark_synthetic

pd.set_option("display.max_colwidth", 140)
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)


REPO_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows
DATA_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows/src/data


In [6]:
DATASET_PATH = DATA_ROOT / "synthetic" / "dataset_sandbox_type2.json"

# GAP
GAP_STEP1_METHOD = "lp"
GAP_ITER = 120

# Genetic
GA_POPULATION_SIZE = 60
GA_GENERATIONS = 120
GA_ELITE_SIZE = 4
GA_SEED = 42

# Live progress in notebook output
SHOW_ALGO_PROGRESS = True
SHOW_SOLVER_DETAILS = False

from time import perf_counter

def make_progress_logger(enabled: bool):
    if not enabled:
        return None
    t0 = perf_counter()
    def _log(message: str) -> None:
        dt = perf_counter() - t0
        print(f"[+{dt:7.1f}s] {message}", flush=True)
    return _log

progress_log = make_progress_logger(SHOW_ALGO_PROGRESS)

benchmark_df = benchmark_synthetic(
    dataset_path=DATASET_PATH,
    gap_step1_method=GAP_STEP1_METHOD,
    gap_iter=GAP_ITER,
    ga_population_size=GA_POPULATION_SIZE,
    ga_generations=GA_GENERATIONS,
    ga_elite_size=GA_ELITE_SIZE,
    ga_seed=GA_SEED,
    show_progress=SHOW_SOLVER_DETAILS,
    progress_hook=progress_log,
)



# Save benchmark artifact to local JSON
import json
from datetime import datetime

LOCAL_OUT_DIR = REPO_ROOT / "notebooks" / "local" / "synthetic_data"
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULT_JSON_PATH = LOCAL_OUT_DIR / f"benchmark_{RUN_TAG}.json"

records = benchmark_df.where(pd.notna(benchmark_df), None).to_dict(orient="records")
artifact = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": "synthetic_3algo_benchmark.ipynb",
    "dataset_path": str(DATASET_PATH),
    "config": {
        "gap_step1_method": GAP_STEP1_METHOD,
        "gap_iter": GAP_ITER,
        "ga_population_size": GA_POPULATION_SIZE,
        "ga_generations": GA_GENERATIONS,
        "ga_elite_size": GA_ELITE_SIZE,
        "ga_seed": GA_SEED,
        "show_algo_progress": SHOW_ALGO_PROGRESS,
        "show_solver_details": SHOW_SOLVER_DETAILS,
    },
    "results": records,
}
RESULT_JSON_PATH.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", RESULT_JSON_PATH)

benchmark_df[[
    "algorithm",
    "feasible",
    "all_checks_ok",
    "assigned_routes",
    "unassigned_tasks",
    "active_agents",
    "transport_work_ton_km",
    "total_km",
    "deadhead_km",
    "deadhead_share_pct",
    "total_hours",
    "runtime_sec",
]]

,algorithm,feasible,all_checks_ok,assigned_routes,unassigned_tasks,active_agents,transport_work_ton_km,runtime_sec
0,milp,True,True,18,0,18,148.075,8.676
1,gap_vrp,True,True,18,0,7,148.075,18.449
2,genetic,True,True,18,0,14,148.075,31.155


In [ ]:
# Детали проверок и параметров по каждому алгоритму
benchmark_df[["algorithm", "checks", "step1_method", "gap_iter", "population_size", "generations", "elite_size"]]


## Как читать benchmark

- `transport_work_ton_km`: сервисная тонна-километровая работа (часто совпадает при фиксированных source->destination).
- `total_km`, `deadhead_km`, `deadhead_share_pct`, `total_hours`: операционные метрики, зависящие от распределения задач по агентам.
- Для сравнения стратегий назначения в первую очередь смотрите на `total_km` и `deadhead_share_pct` при `all_checks_ok=True`.



In [ ]:
from flowopt.reporting import solution_breakdown_tables
from IPython.display import Markdown, display

MAX_AGENTS_TO_SHOW = 30
MAX_TASK_IDS_IN_CELL = 12
MAX_TASK_ROWS_TO_SHOW = 400

for _, row in benchmark_df.iterrows():
    algo = row.get("algorithm", "unknown")
    tables = solution_breakdown_tables(
        row,
        max_agents=MAX_AGENTS_TO_SHOW,
        max_task_ids=MAX_TASK_IDS_IN_CELL,
    )

    display(Markdown(f"### {algo}: решение по агентам"))
    display(tables["summary"])

    if tables["agents"].empty:
        print("No active agents in solution")
        continue

    display(tables["agents"])
    display(Markdown(f"**{algo}: задача -> агент (первые {MAX_TASK_ROWS_TO_SHOW} строк)**"))
    display(tables["tasks"].head(MAX_TASK_ROWS_TO_SHOW))
